# Nexa HU — saját adatokkal tanított modell

Ez a notebook a **saját Nexa modellt** készíti. A nyílt `Qwen/Qwen3-0.6B` bázist
**két szakaszban** tanítjuk magyar adatokon, majd összeolvasztjuk a súlyokat,
ONNX-be exportáljuk, 4 bitesre kvantáljuk, és a saját Hugging Face repónkra
töltjük. Az eredmény telefoton, offline is fut a Nexa AI appban.

**A tanítás két szakasza**

1. **Folyamatos magyar előképzés** — magyar Wikipédia cikkekből vett szövegblokkok;
   ez adja a magyar nyelvtant, szókincset és tényanyagot.
2. **Utasítás-követő finomhangolás** — a saját mintáink, az Aya magyar párjai és
   az OPUS-100 fordítási párok; ez tanítja meg a „válaszolj röviden, de fejezd be”
   viselkedésre.

**Mit jelent és mit nem jelent a „saját modell”**

- Igen: a súlyokat a mi adatainkkal képezzük, a modell magyarul válaszol,
  `Nexa AI`-ként azonosítja magát, és a saját Hugging Face repónkon van.
- Nem: nulláról, saját architektúrával betanított LLM. Az több ezer GPU-óra és
  hatalmas korpusz lenne, és kis telefoni modellként sokkal gyengébb lenne.
  A bázis nyílt (Apache 2.0), így a finomhangolt változat szabadon terjeszthető.
- A „mindent egyszerre” élményt az app adja: a szöveges agy ez a modell, a hang-
  és képfeladatokat az app külön, szintén helyi modellekre irányítja.

**Futtatás:** Colabban `Futtatás → Futtatókörnyezet módosítása → T4 GPU`, majd
celláról cellára végig. A bővített tanítás kb. **60–90 perc**.

**Előfeltétel:** Hugging Face fiók és *write* token, amit a Colab „Secrets”
paneljén `HF_TOKEN` néven tárolsz (vagy a notebook kér be).

In [ ]:
# 0. Környezet telepítése (Colab T4 GPU)
!pip -q install -U "transformers>=4.51,<5" "peft>=0.13,<1" "datasets>=3,<4" "accelerate>=1,<2" "bitsandbytes>=0.45" "optimum[onnxruntime]>=1.24,<2" "onnx>=1.17,<2" "onnxruntime>=1.20,<2" "sentencepiece" "protobuf" "huggingface_hub>=0.30,<1"

import torch, transformers, peft, datasets, optimum, onnx, onnxruntime
print("transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NINCS")
assert torch.cuda.is_available(), "Kapcsold be a GPU-t: Futtatás -> Futtatókörnyezet módosítása -> T4 GPU"

In [ ]:
# 1. Beállítások — itt módosítsd a cél repót, ha nem az alapértelmezettet használod
import os

BASE_MODEL     = "Qwen/Qwen3-0.6B"
OUTPUT_REPO    = "sziriusz-blackalt/nexa-hu"   # <-- a saját Hugging Face modellrepód

MAX_LEN        = 768      # egy tanítási blokk hossza tokenben
RAW_CHUNKS     = 4000     # magyar Wikipédiából vett blokkok száma (~3M token)
PRETRAIN_EPOCHS = 1       # 1. szakasz: folyamatos magyar előképzés
PRETRAIN_LR    = 2e-4
EPOCHS         = 2        # 2. szakasz: utasítás-követő finomhangolás
LEARN_RATE     = 1e-4
LORA_R         = 32       # nagyobb adapter = több tanulható tudás
LORA_ALPHA     = 64
BATCH          = 4
GRAD_ACCUM     = 8

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Hugging Face write token: ")

import huggingface_hub
huggingface_hub.login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Cél repo:", OUTPUT_REPO)

In [ ]:
# 2. Saját magyar tanítóadatok — ezeket mi írtuk, a Nexa AI viselkedésére szabva
# A rövid, de teljes válaszok szándékosak: az app sem csonkítja a mondatot.
import json

SEED_JSONL = r"""
{"instruction":"Szia, hogy vagy?","response":"Szia! Jól vagyok, köszönöm. Neked hogy megy?"}
{"instruction":"Hogy vagy?","response":"Jól vagyok, köszönöm! Szólj, ha segíthetek valamiben."}
{"instruction":"Jó reggelt!","response":"Jó reggelt! Mivel tudok segíteni ma?"}
{"instruction":"Jó estét!","response":"Jó estét! Mit szeretnél megbeszélni?"}
{"instruction":"Köszönöm szépen!","response":"Szívesen! Bármikor visszatérhetsz."}
{"instruction":"Ki vagy te?","response":"Nexa AI vagyok, egy magyar nyelvű segéd, amely a telefonodon, internet nélkül is működik."}
{"instruction":"Milyen modell vagy?","response":"A Nexa HU modell vagyok: egy kis magyar nyelvű modell, amit saját adatokkal finomhangoltak."}
{"instruction":"Meg tudod nyitni a kamerát?","response":"A képfeldolgozást az app külön modellje végzi a Képek menüben. Itt a csevegésre tudok válaszolni."}
{"instruction":"El tudod olvasni a hangomat?","response":"A hangfelismerést az app Hang menüje végzi. Ide írhatod vagy mondhatod át a kérdést."}
{"instruction":"Használsz internetet?","response":"Nem küldök adatot sehova. A modell a telefonodon fut, a letöltés után offline is működik."}
{"instruction":"Gyűjtöd a személyes adataimat?","response":"Nem. Minden csevegés csak a készülékeden marad, és nem kerül szerverre."}
{"instruction":"Tudsz angolul?","response":"Igen, tudok angolul is, de a magyar válaszokban vagyok a legjobb."}
{"instruction":"Milyen nyelven válaszolsz?","response":"Alapból magyarul, röviden és természetesen. Ha kéred, angolra is váltok."}
{"instruction":"Válaszolj röviden!","response":"Értem, rövid maradok."}
{"instruction":"Beszélj részletesebben!","response":"Persze. Melyik részt szeretnéd részletesebben?"}
{"instruction":"Mi a fővárosa Magyarországnak?","response":"Magyarország fővárosa Budapest."}
{"instruction":"Mennyi 12 szorozva 7?","response":"12 × 7 = 84."}
{"instruction":"Mennyi 15% -a a 200-nak?","response":"A 200 15%-a 30."}
{"instruction":"Oldd meg: 2x + 4 = 10","response":"2x + 4 = 10, tehát 2x = 6, így x = 3."}
{"instruction":"Mi a különbség a méter és a kilométer között?","response":"Egy kilométer ezer méter. A méter az alapegység, a kilométer a nagyobb távolságoké."}
{"instruction":"Foglalj össze egy mondatban: A nap süt, az ég kék, madarak énekelnek.","response":"Szép nappal van: süt a nap, kék az ég és madarak énekelnek."}
{"instruction":"Foglalj össze röviden: A cikk a magyar fizikusokról szól, különösen Szilárd Leóról és Wigner Jenőről.","response":"A cikk magyar fizikusokat mutat be, köztük Szilárd Leót és Wigner Jenőt."}
{"instruction":"Fordítsd magyarra: Good morning, how are you?","response":"Jó reggelt, hogy vagy?"}
{"instruction":"Fordítsd magyarra: I would like a coffee, please.","response":"Egy kávét kérnék, köszönöm."}
{"instruction":"Fordítsd magyarra: The train leaves at eight o'clock.","response":"A vonat nyolckor indul."}
{"instruction":"Fordítsd angolra: Jó napot kívánok!","response":"Good day to you!"}
{"instruction":"Fordítsd angolra: Hol a legközelebbi bolt?","response":"Where is the nearest shop?"}
{"instruction":"Fordítsd angolra: Köszönöm a segítséget.","response":"Thank you for your help."}
{"instruction":"Írj egy rövid szép mondatot az őszről!","response":"Az ősz aranyszínű levelei csendesen hullanak a hidegebb szélben."}
{"instruction":"Adj címet egy nyári tengerparti képnek!","response":"Nyári tengerpart: kék víz és meleg homok."}
{"instruction":"Javasolj egy rövid célt a tanuláshoz!","response":"Tanulj ma 20 percet, és ismételj át két dolgot, amit már ismersz."}
{"instruction":"Hogyan kezdjek el programozni?","response":"Kezdd Pythonnal: tanuld meg a változókat, a feltételeket és a ciklusokat, majd írj kis gyakorlóprogramokat."}
{"instruction":"Mi a különbség a lista és a szótár között Pythonban?","response":"A lista sorrendben tárol értékeket indexekkel, a szótár pedig kulcs-érték párokat."}
{"instruction":"Írj Python függvényt, amely összead két számot!","response":"def osszead(a, b):\n    return a + b\n\nHasználat: osszead(2, 3) -> 5"}
{"instruction":"Írj JavaScript kódot, ami kiírja a „Hello” szöveget a konzolra!","response":"console.log('Hello');"}
{"instruction":"Hogyan hozzam létre a HTML alapstruktúráját?","response":"<!DOCTYPE html>\n<html>\n<head><title>Cím</title></head>\n<body>Tartalom</body>\n</html>"}
{"instruction":"Mire való a CSS?","response":"A CSS az oldal megjelenését írja le: színeket, betűket, térközöket és elrendezést."}
{"instruction":"Találd meg a hibát: print(1 + )","response":"A zárójelben hiányzik a második szám. Javítva: print(1 + 2)"}
{"instruction":"Magyarázd el röviden, mi a JSON!","response":"A JSON adatcsere-formátum: kulcsokból és értékekből álló szöveg, amit a programok könnyen beolvasnak."}
{"instruction":"Mi az a SQL SELECT?","response":"A SELECT utasítás adatokat olvas ki egy adatbázis táblájából."}
{"instruction":"Írj SQL lekérdezést, ami kiválasztja az összes nevet a users táblából!","response":"SELECT name FROM users;"}
{"instruction":"Mi a különbség a = és a == között JavaScriptben?","response":"A = értéket ad, a == pedig értéket hasonlít össze (típuskonverzióval)."}
{"instruction":"Hogyan tároljak szöveget Python változóban?","response":"Így: nev = 'Magyarorszag' vagy nev = \"Magyarorszag\"."}
{"instruction":"Mi a Fibonacci-sorozat?","response":"Olyan számsor, ahol minden tag az előző kettő összege: 0, 1, 1, 2, 3, 5, 8."}
{"instruction":"Sorolj fel három begyűrűzőt Magyarországról!","response":"Szilárd Leó, Wigner Jenő és Teller Ede."}
{"instruction":"Miért fontos az offline működés?","response":"Mert így a modell nem küld adatot szerverre, és repülőn vagy gyenge hálózat mellett is használható."}
{"instruction":"Le tudod fordítani ezt: „The weather is nice today.”","response":"Ma szép az idő."}
{"instruction":"Mi a magyarázata annak, hogy esik az eső?","response":"A levegőben lévő vízgőz lehűl, cseppekké áll össze, és a gravitáció leejti a talajra."}
{"instruction":"Hogyan írjak rövid összefoglalót egy hosszú szövegből?","response":"Keresd meg a főállítást, emeld ki a három legfontosabb tényt, és fogalmazd egy-két mondatba."}
{"instruction":"Segíts megfogalmazni egy udvarias e-mail kezdetét!","response":"„Tisztelt Címzett!\n\nRemélem, jól van. Az alábbiakban röviden leírom a kérdésemet.”"}
{"instruction":"Mi a teendő, ha az app üres választ ad?","response":"Próbáld újra rövidebb kérdéssel, és ellenőrizd a Beállításokban, hogy be van-e töltve a Nova modell."}
{"instruction":"Melyik modellt válasszam gyors válaszhoz?","response":"A ⚡ Szikra gyors, de magyarul hibás lehet. Értelmes válaszhoz a ✨ Novát vagy a saját Nexa HU modellt használd."}
{"instruction":"Hogyan töltsök le modellt offline használathoz?","response":"A Beállítások → Modellek menüben nyomj a „Letöltés” gombra, majd kapcsold be a szigorú offline módot."}
{"instruction":"Mi történik a csevegéseimmel?","response":"A csevegéseid a készülékeden maradnak. Ideiglenes csevegésnél nem is kerülnek elmentésre."}
{"instruction":"Újra tudod írni ezt rövidebben: „Szeretném megkérni, hogy legyen szíves segíteni nekem.”","response":"„Segítenél nekem, kérlek?”"}
{"instruction":"Mi a különbség az összefoglaló és a fordítás között?","response":"Az összefoglaló rövidíti a tartalmat, a fordítás pedig más nyelvre alakítja, a jelentés megtartásával."}
{"instruction":"Adj ötleteket egy rövid magyar vershez!","response":"Írj négy sort a szélről és egy elfelejtett levélről; a végén egy váratlan szóval zárd."}
{"instruction":"Mit jelent a „kód mód” az appban?","response":"A kód mód a csevegés egyik stílusa: kódot ír, magyaráz vagy javít, és nem külön képernyő."}
{"instruction":"Hogyan kérjek kódot tőled?","response":"Írd meg, milyen nyelven és mit szeretnél, például: „Írj Python függvényt, ami rendez egy listát.”"}
{"instruction":"Miért nem válaszolsz hosszú cikkekre?","response":"Hosszú szövegre is válaszolok: először a lényeget mondom, majd kérésre részletezem. Ha rövidebb kell, kérj összefoglalót."}
{"instruction":"Ellenőrizd a mondatot: „én szeretem a könyveket.”","response":"„Én szeretem a könyveket.” — nagy kezdőbetűt írtam az éN helyett."}
{"instruction":"Mi a helyes: „hol vagy” vagy „hol vagyon”?","response":"A helyes: „Hol vagy?” A „vagyon” birtokot jelent."}
{"instruction":" Hogyan mondjam magyarul: „I am fine, thanks.”","response":"„Jól vagyok, köszönöm.”"}
{"instruction":"Mennyi 9 plusz 8?","response":"9 + 8 = 17."}
{"instruction":"Melyik nagyobb: 0.5 vagy 0.45?","response":"A 0.5 nagyobb, mint a 0.45."}
{"instruction":"Írj egy rövid vicces mondatot!","response":"A kávém ma reggel olyan erős volt, hogy a szék is felállt."}
{"instruction":"Hogyan készítek elő egy beszédet?","response":"Válassz egy fő üzenetet, írj három pontot alá, és gyakorold rövidre szabva."}
{"instruction":"Mi a fontsorrend a biztonságban?","response":"Előbb a személyes adatok védelme, aztán a hálózati kapcsolat, végül a kényelem."}
"""

seed = [json.loads(line) for line in SEED_JSONL.strip().splitlines() if line.strip()]
print("Saját magyar példák:", len(seed))

In [ ]:
# 3. Nyilvános magyar adatok: utasítások + nyers szövegek
from datasets import Dataset, load_dataset

records = list(seed)
raw_texts = []

# Aya Dataset – magyar, emberi utasítás-válasz párok
try:
    aya = load_dataset("CohereLabs/aya_dataset", split="train")
    aya_hu = aya.filter(lambda r: r["language_code"] == "hun")
    for r in aya_hu:
        ins = (r.get("inputs") or "").strip()
        res = (r.get("targets") or "").strip()
        if ins and res:
            records.append({"instruction": ins, "response": res})
    print("Aya magyar példák:", len(aya_hu))
except Exception as e:
    print("Aya kihagyva:", str(e)[:180])

# OPUS-100 angol-magyar mondatpárok -> fordítási utasítások
try:
    opus = load_dataset("Helsinki-NLP/opus-100", "en-hu", split="train[:12000]")
    added = 0
    for r in opus:
        tr = r["translation"]
        en, hu = tr["en"].strip(), tr["hu"].strip()
        if not en or not hu or len(en) > 220 or len(hu) > 220:
            continue
        records.append({"instruction": "Fordítsd magyarra: " + en, "response": hu})
        records.append({"instruction": "Translate to English: " + hu, "response": en})
        added += 2
    print("OPUS fordítási példák:", added)
except Exception as e:
    print("OPUS kihagyva:", str(e)[:180])

# Magyar Wikipédia – nyers szöveg az 1. szakaszbeli folyamatos előképzéshez
try:
    wiki = load_dataset("wikimedia/wikipedia", "20231101.hu", split="train[:20000]")
    for r in wiki:
        text = (r.get("text") or "").strip()
        if len(text) > 200:
            raw_texts.append(text)
    print("Magyar Wikipédia cikkek:", len(raw_texts))
except Exception as e:
    print("Wikipédia kihagyva:", str(e)[:180])

# Tisztítás, duplikátumok eltávolítása (utasítások)
seen, clean = set(), []
for r in records:
    ins, res = r["instruction"].strip(), r["response"].strip()
    if not ins or not res or len(ins) > 1500 or len(res) > 1500:
        continue
    key = (ins, res)
    if key in seen:
        continue
    seen.add(key)
    clean.append({"instruction": ins, "response": res})

print("Összes utasítási példa:", len(clean))
print("Nyers magyar szövegek:", len(raw_texts))
ds = Dataset.from_list(clean)
ds[0]

In [ ]:
# 4. Utasítások becsomagolása Qwen3 sablonba, „thinking” kikapcsolva
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
SYSTEM = ("Te Nexa AI vagy, egy magyar nyelvű, offline segéd. "
          "Válaszolj hibátlan, természetes magyarul. Légy rövid, de teljes: "
          "fejezd be a gondolatot. Ne találj ki tényeket. Ha nem tudod, mondd, hogy nem tudod.")

def to_example(r):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": r["instruction"]},
        {"role": "assistant", "content": r["response"]},
    ]
    # A Qwen3 alapból <think> blokkal indulna; a Nexa app is enable_thinking=false-t használ.
    prompt = tok.apply_chat_template(messages[:2], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    full = tok.apply_chat_template(messages, tokenize=False,
                                   add_generation_prompt=False, enable_thinking=False)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"]
    if f_ids[:len(p_ids)] != p_ids:
        p_ids = f_ids[:max(1, len(f_ids) - 1)]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    ids = f_ids[:MAX_LEN]
    labels = labels[:MAX_LEN]
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}

formatted = ds.map(to_example, remove_columns=ds.column_names)
formatted = formatted.filter(lambda r: len(r["input_ids"]) > 8)
print("Betanításra kész utasítások:", len(formatted))
print(tok.decode(formatted[0]["input_ids"][:60]))

In [ ]:
# 5. LoRA tanítás két szakaszban: (1) magyar szöveg, (2) utasítások
import torch
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

class Collator:
    def __call__(self, features):
        n = max(len(f["input_ids"]) for f in features)
        pad = tok.pad_token_id if tok.pad_token_id is not None else 0
        return {
            "input_ids": torch.tensor([f["input_ids"] + [pad] * (n - len(f["input_ids"])) for f in features]),
            "labels": torch.tensor([f["labels"] + [-100] * (n - len(f["labels"])) for f in features]),
            "attention_mask": torch.tensor([f["attention_mask"] + [0] * (n - len(f["attention_mask"])) for f in features]),
        }

def make_args(stage, lr, epochs):
    return TrainingArguments(
        output_dir=f"/content/nexa-out-{stage}",
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=20,
        save_strategy="no",
        bf16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        report_to=[],
    )

# 1. szakasz: nyers magyar szövegblokkok (tudás, nyelvtan, szókincs)
raw_rows = []
if raw_texts:
    buf = []
    for text in raw_texts:
        buf.extend(tok(text, add_special_tokens=False)["input_ids"])
        while len(buf) >= MAX_LEN and len(raw_rows) < RAW_CHUNKS:
            chunk = buf[:MAX_LEN]
            raw_rows.append({
                "input_ids": list(chunk),
                "labels": list(chunk),
                "attention_mask": [1] * MAX_LEN,
            })
            buf = buf[MAX_LEN:]
        if len(raw_rows) >= RAW_CHUNKS:
            break
    print("Nyers szövegblokkok:", len(raw_rows))

if raw_rows:
    raw_ds = Dataset.from_list(raw_rows)
    print("1. szakasz: folyamatos magyar előképzés")
    trainer = Trainer(model=model, args=make_args("raw", PRETRAIN_LR, PRETRAIN_EPOCHS),
                      train_dataset=raw_ds, data_collator=Collator())
    trainer.train()
    del trainer, raw_ds
    torch.cuda.empty_cache()

# 2. szakasz: rövid, teljes magyar válaszok utasításra
print("2. szakasz: utasítás-követő finomhangolás")
trainer = Trainer(model=model, args=make_args("instruct", LEARN_RATE, EPOCHS),
                  train_dataset=formatted, data_collator=Collator())
trainer.train()
del trainer
torch.cuda.empty_cache()

In [ ]:
# 6. Ellenőrzés magyarul, majd a LoRA összeolvasztása
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Ki vagy te és miben segíthetsz?"},
]
prompt = tok.apply_chat_template(messages, tokenize=False,
                                 add_generation_prompt=True, enable_thinking=False)
inputs = tok(prompt, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False, pad_token_id=tok.pad_token_id)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

merged = model.merge_and_unload()
merged.config.use_cache = True
MERGED_DIR = "/content/nexa-merged"
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")
tok.save_pretrained(MERGED_DIR)

# Generálási beállítások: teljes, de nem túl hosszú magyar válasz
import json
gc_path = os.path.join(MERGED_DIR, "generation_config.json")
gc = json.load(open(gc_path, encoding="utf-8")) if os.path.exists(gc_path) else {}
gc.update({
    "max_new_tokens": 256,
    "do_sample": False,
    "temperature": 0.2,
    "top_p": 0.8,
    "eos_token_id": tok.eos_token_id,
    "pad_token_id": tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
})
json.dump(gc, open(gc_path, "w", encoding="utf-8"), indent=2)
print("Összeolvasztva:", MERGED_DIR)

In [ ]:
# 7. ONNX export — a telefonos Transformers.js által ismert struktúra
import shutil
from pathlib import Path
from optimum.exporters.onnx import main_export

ONNX_DIR = "/content/nexa-onnx"
shutil.rmtree(ONNX_DIR, ignore_errors=True)

main_export(
    model_name_or_path=MERGED_DIR,
    output=ONNX_DIR,
    task="text-generation-with-past",
    device="cpu",
    fp16=False,
    no_post_process=True,
)

# A tokenizer és config fájlok a repo gyökerébe kerülnek (az app innen olvassa)
for name in ["config.json", "tokenizer.json", "tokenizer_config.json",
             "generation_config.json", "special_tokens_map.json"]:
    src = os.path.join(MERGED_DIR, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(ONNX_DIR, name))

print("Exportált fájlok:")
for p in sorted(Path(ONNX_DIR).rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(ONNX_DIR), round(p.stat().st_size / 1048576, 1), "MB")

In [ ]:
# 8. 4 bites kvantálás -- ez a telefonra való (~350-450 MB), pontosan az app altal kert "q4" tipus
from pathlib import Path
import shutil
from onnxruntime.quantization import matmul_nbits_quantizer

onnx_dir = Path(ONNX_DIR) / "onnx"
model_path = onnx_dir / "model.onnx"
q_path = onnx_dir / "model_q4.onnx"

quant = matmul_nbits_quantizer.MatMulNBitsQuantizer(
    model=str(model_path),
    block_size=32,
    is_symmetric=True,
    bits=4,
)
quant.process()
quant.model.save(str(q_path))

# A nyers fp32 modell nem kell a telefonra
model_path.unlink(missing_ok=True)
(model_path.with_name(model_path.name + "_data")).unlink(missing_ok=True)

# A Transformers.js egyes verzioi dtype=q4 mellett is a model.onnx nevet keresik
shutil.copy(q_path, model_path)

size_mb = q_path.stat().st_size / 1048576
print("model_q4.onnx:", round(size_mb, 1), "MB")
assert size_mb < 900, "Tul nagy modell -- ellenorizd a kvant-alast"

In [ ]:
# 9. Gyors ellenőrzés: nem üres-e a kimenet, van-e KV cache
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

check_tok = AutoTokenizer.from_pretrained(ONNX_DIR)
check_prompt = check_tok.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Szia!"}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)
ids = check_tok(check_prompt, return_tensors="np")["input_ids"].astype(np.int64)
mask = np.ones_like(ids)

session = ort.InferenceSession(str(q_path), providers=["CPUExecutionProvider"])
inputs = {i.name: i for i in session.get_inputs()}
feed = {"input_ids": ids, "attention_mask": mask}
if "position_ids" in inputs:
    feed["position_ids"] = np.arange(ids.shape[1], dtype=np.int64).reshape(1, -1)
for name, inp in sorted(inputs.items()):
    if name.startswith("past_key_values"):
        shape = [d if isinstance(d, int) else 0 for d in inp.shape]
        if shape and shape[0] == 0:
            shape[0] = 1
        feed[name] = np.zeros(shape, dtype=np.float32)

logits = session.run([o.name for o in session.get_outputs()], feed)[0]
assert np.isfinite(logits).all(), "A modell NaN/Inf erteket ad"
assert logits.max() > logits.min(), "Allando logits -- a modell nem mukodik"
first = check_tok.decode([int(np.argmax(logits[0, -1, :]))], skip_special_tokens=True)
print("Kovetkezo token:", repr(first))
print("KV cache kimenetek:", sum(1 for o in session.get_outputs() if o.name.startswith("present")))
print("ELLENORZES OK")

In [ ]:
# 10. Feltöltés a Hugging Face-re
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=OUTPUT_REPO, repo_type="model", exist_ok=True)

readme = f"""---
base_model: {BASE_MODEL}
library_name: transformers.js
tags: [onnx, transformers.js, hungarian, nexa, q4]
---
# {OUTPUT_REPO.split('/')[-1]}

Két szakaszban, magyar adatokkal tanított Nexa HU modell:
folyamatos magyar előképzés (magyar Wikipédia), majd utasítás-követő
finomhangolás (saját minták, Aya hu, OPUS-100). A súlyokat a Nexa AI
projekt tanította be; a modell telefonon, offline fut.

- Bázis: `{BASE_MODEL}` (Apache 2.0)
- Tanítás: LoRA r={LORA_R}, `enable_thinking=false`
- Formátum: ONNX, 4 bites `MatMulNBits` (`onnx/model_q4.onnx`)
- Használat a Nexa AI-ban: Beállítások → *Saját Nexa modell* → `{OUTPUT_REPO}`

A modell nem küld adatot szerverre.
"""
with open(os.path.join(ONNX_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=OUTPUT_REPO,
    repo_type="model",
    commit_message="Nexa HU: ketszakaszos magyar tanitas (q4 ONNX)",
)
print("Kesz: https://huggingface.co/" + OUTPUT_REPO)

## Készen állsz — így használd az appban

1. A Hugging Face repo nyilvános legyen.
2. Nexa AI → **Beállítások → Modellek → Saját Nexa modell**: írd be a fenti repo nevedet
   (pl. `sziriusz-blackalt/nexa-hu`).
3. Nyomj a megjelenő **🧠 Nexa HU** kártya *Letöltés* gombjára.
4. A csevegés **Modell** gombjából válaszd a **🧠 Nexa HU**-t.

Ha a letöltés nem indul, ellenőrizd a repo nevét és hogy az `onnx/model_q4.onnx`
fájl valóban fent van-e. A beépített Nova/Szikra modellek változatlanul elérhetők.